# 2c — $[X_f, X_g] = X_{\{f, g\}}$

**Problem (c).** Aynı symplectic kurulumda ($d\omega = 0$, $\iota_{X_h}\omega = dh$, $\{f,g\} := \pi(df, dg)$) iki Hamiltonian vektör alanının Lie bracket'i yine Hamiltonian'dır ve karşılık gelen fonksiyon Poisson bracket'tir:

$$
[X_f, X_g] = X_{\{f, g\}}.
$$

Bu bir **operator-level** eşitlik — iki vektör alanının özdeşliği. Sistem şu anda alternating $p$-form evaluation node'una sahip olmadığı için iki vektör alanını doğrudan eşitlemeyi engine'e yaptıramıyoruz (bkz. Faz 12). Bu yüzden **iki-adımlı strateji** izliyoruz:

1. **Engine ile** — her iki tarafın $\omega$'ya ι ile etkisinin aynı olduğunu göster: $\iota_{[X_f, X_g]}\omega = d(X_f(g))$.
2. **Markdown ile** — non-degeneracy ve (b) sonucunu çağırarak $[X_f, X_g] = X_{\{f, g\}}$ operator-level sonucunu kapat.

$\omega$ non-degenerate olduğu için bir Hamiltonian vektör alanı onun $\omega$-contraction'ı tarafından tek değerli biçimde belirlenir; bu notebook'un ilk adımını engine kapatır, ikinci adım operator-level mantıksal sonuçtur.

## Strateji — element-level hedef

Yola çıktığımız kimlik Cartan kalkülüs'ün **$\iota$-$L$ değişme ilişkisi** (a.k.a. `lie_iota`):

$$
\iota_{[X, Y]} = L_X \iota_Y - \iota_Y L_X.
$$

$(X, Y) = (X_f, X_g)$ alalım ve her iki tarafı $\omega$'ya uygulayalım:

$$
\iota_{[X_f, X_g]}\omega = L_{X_f}(\iota_{X_g}\omega) - \iota_{X_g}(L_{X_f}\omega).
$$

Sağ tarafta:

| Parça | İndirgeme | Kaynak |
|---|---|---|
| $\iota_{X_g}\omega = dg$ | Hamiltonian defining relation (g) | Aksiyom A3 |
| $L_{X_f}(dg) \rightsquigarrow d(X_f(g))$ | Cartan magic + pairing + $d^2 = 0$ | Yerleşik |
| $L_{X_f}\omega \rightsquigarrow 0$ | Cartan magic + $\iota_{X_f}\omega = df$ + $d\omega = 0$ | 2a'nın kendisi |
| $\iota_{X_g}(0) = 0$ | $\iota_X$ 0-form üzerinde | Yerleşik |

Bu üç aksiyom (A3, A4, A8) + `lie_iota` özelçesi (A6) engine'in zinciri kapatması için yeter. Hedef: $\iota_{[X_f, X_g]}\omega = d(X_f(g))$. Dikkat: sağ taraf **(b)'nin sonucu** $\{f, g\} = X_f(g)$ ile $d\{f,g\}$'ye denktir.

In [1]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Kurulum

- $f, g$ — fonksiyon (0-form).
- $\omega$ — symplectic 2-form (Graded(2)).
- $X_f, X_g$ — Hamiltonian vector fields (derece-0 `Derivation`).
- $L_{X_f}$ — Cartan-mode Lie türevi, `lie_derivative(X_f)` çağrısıyla üretilir (engine'e `LieDerivativeCartanDefinition` ile bağlı).
- $[X_f, X_g]$ **açık şekilde** `Sum(Product(X_f, X_g), Neg(Product(X_g, X_f)))` olarak kurulur. İki alan da derece-0 olduğu için graded commutator'sı eşit işaretli ($AB - BA$) — bu, `Commutator.expand()` sonucuna denk.
- $\iota_{[X_f, X_g]}$ — `interior(lie_bracket, name="ι_[X_f,X_g]")`; içşeriği `InteriorProduct`'ın `vector_field` attribute'u olarak saklar.

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Expr, Integer, Neg, Product, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
g = Symbol("g")
reg.declare(f, Graded(degree=0))
reg.declare(g, Graded(degree=0))

omega = Symbol("ω")
reg.declare(omega, Graded(degree=2))

X_f = Derivation("X_f", degree=0)
X_g = Derivation("X_g", degree=0)
L_Xf = lie_derivative(X_f)  # cartan mode (default)

# [X_f, X_g] as an explicit Expr (both degree 0 → even parity).
lie_bracket = Sum(Product(X_f, X_g), Neg(Product(X_g, X_f)))
iota_bracket = interior(lie_bracket, name="ι_[X_f,X_g]")

print("f, g           :", f, g)
print("ω              :", omega, " degree =", reg.get(omega, Graded).degree)
print("X_f, X_g       :", X_f, X_g)
print("L_{X_f}        :", L_Xf)
print("[X_f, X_g]     :", lie_bracket)
print("ι_{[X_f,X_g]}  :", iota_bracket)

f, g           : f g
ω              : ω  degree = 2
X_f, X_g       : X_f X_g
L_{X_f}        : L_X_f
[X_f, X_g]     : ((X_f * X_g) + (-(X_g * X_f)))
ι_{[X_f,X_g]}  : ι_[X_f,X_g]


## 2. Problem-aksiyomları

Dört `Definition` kayıt ediyoruz:

- **A6 — `lie_iota` özelliği** — $\iota_{[X_f, X_g]}(\omega) = L_{X_f}(\iota_{X_g}\omega) - \iota_{X_g}(L_{X_f}\omega)$. 
  `CartanCalculus.lie_iota` genel operator-level kimliğin (matchable) formu; burada sadece $\omega$'ya etkidiği şiçi element-level üzerine pin'liyoruz ki engine bottom-up ulaşsın.
- **A8 — $d\omega = 0$** — symplectic closedness (2a'daki gibi).
- **A4 — $\iota_{X_f}\omega = df$** — Hamiltonian defining relation (f).
- **A3 — $\iota_{X_g}\omega = dg$** — Hamiltonian defining relation (g).

A6 aksiyomsal görünsün — Cartan kalkülüs altyapısında bir theorem; `CartanCalculus.verify("lie_iota", ...)` işlenmiş halde verir. Bu notebook'ta onu kısa yoldan aksiyom gibi ekliyoruz; "unrolled" versiyonu ileride istersek `foundational` modla açılır.

In [3]:
class LieIotaOnOmega(Definition):
    """A6 (specialized): ι_{[X_f,X_g]}(ω) = L_{X_f}(ι_{X_g}ω) - ι_{X_g}(L_{X_f}ω)."""
    name = "ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        op = expr.op
        if not isinstance(op, InteriorProduct):
            return False
        if op.vector_field != lie_bracket:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        z = expr.arg
        return Sum(
            Act(L_Xf, Act(interior(X_g), z)),
            Neg(Act(interior(X_g), Act(L_Xf, z))),
        )


class DOmegaClosed(Definition):
    """A8: dω = 0."""
    name = "dω = 0 (symplectic)"

    def matches(self, expr):
        return (
            isinstance(expr, Act)
            and isinstance(expr.op, ExteriorDerivative)
            and expr.op == d
            and expr.arg == omega
        )

    def rewrite(self, expr):
        return Integer(0)


class IotaXfOmegaIsDf(Definition):
    """A4: ι_{X_f} ω = df."""
    name = "ι_{X_f} ω = df"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_f:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, f)


class IotaXgOmegaIsDg(Definition):
    """A3: ι_{X_g} ω = dg."""
    name = "ι_{X_g} ω = dg"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_g:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, g)


for axiom in (LieIotaOnOmega, DOmegaClosed, IotaXfOmegaIsDf, IotaXgOmegaIsDg):
    print("-", axiom.name)

- ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω
- dω = 0 (symplectic)
- ι_{X_f} ω = df
- ι_{X_g} ω = dg


## 3. Engine

`default_engine` Cartan magic'i, $d^2=0$'ı, pairing'i ($\iota_X(df) = X(f)$) ve $\iota_X(f)=0$'ı zaten taşır. Dört problem-aksiyomunu üzerine ekliyoruz.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(LieIotaOnOmega())
engine.register(DOmegaClosed())
engine.register(IotaXfOmegaIsDf())
engine.register(IotaXgOmegaIsDg())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 12 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω
 - dω = 0 (symplectic)
 - ι_{X_f} ω = df
 - ι_{X_g} ω = dg


## 4. Hedef ve ispat

**Element-level hedef** (engine kapatacak):

$$
\iota_{[X_f, X_g]}\omega = d(X_f(g)).
$$

Sağ taraftaki $d(X_f(g))$ — (b)'nin sonucu $\{f, g\} = X_f(g)$ kullanılarak $d\{f, g\}$'ye eşit. Dolayısıyla bu zinciri kapatınca $\iota_{[X_f, X_g]}\omega = d\{f, g\}$ olur; yani $[X_f, X_g]$'in $\omega$-contraction'ı $\{f, g\}$'in Hamiltonian'ının $\omega$-contraction'ı ile aynı.

In [5]:
lhs = Act(iota_bracket, omega)
rhs = Act(d, Act(X_f, g))

print("LHS:", lhs)
print("RHS:", rhs)

chain = ExpandAndSimplify().prove(
    lhs, rhs, registry=reg, engine=engine
)
print(f"\nKAPANDI — {len(chain)} adım.")

LHS: ι_[X_f,X_g](ω)
RHS: d(X_f(g))

KAPANDI — 15 adım.


## 5. İspat zinciri — LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

\begin{align*}
\iota_[X_f,X_g]\!\left(\omega\right) &\to L_{X_f}\!\left(\iota_{X_g}\!\left(\omega\right)\right) - \iota_{X_g}\!\left(L_{X_f}\!\left(\omega\right)\right) && \text{[\ensuremath{\iota}\_{[X\_f,X\_g]}(\ensuremath{\omega}) = L\_Xf \ensuremath{\iota}\_Xg \ensuremath{\omega} - \ensuremath{\iota}\_Xg L\_Xf \ensuremath{\omega}]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_{[X\_f,X\_g]}(\ensuremath{\omega}) = L\_Xf \ensuremath{\iota}\_Xg \ensuremath{\omega} - \ensuremath{\iota}\_Xg L\_Xf \ensuremath{\omega}} \\
\iota_{X_g}\!\left(\omega\right) &\to d\!\left(g\right) && \text{[\ensuremath{\iota}\_{X\_g} \ensuremath{\omega} = dg]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_{X\_g} \ensuremath{\omega} = dg} \\
L_{X_f}\!\left(d\!\left(g\right)\right) &\to \left(d \, \iota_{X_f}\right)\!\left(d\!\left(g\right)\right) + \left(\iota_{X_f} \, d\right)\!\left(d\!\left(g\right)\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
L_{X_f}\!\left(\omega\right) &\to \left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\left(\left(\left(d \, \iota_{X_f}\right)\!\left(d\!\left(g\right)\right) + \left(\iota_{X_f} \, d\right)\!\left(d\!\left(g\right)\right)\right) - \iota_{X_g}\!\left(\left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right)\right)\right) - d\!\left(X_f\!\left(g\right)\right) &\to \left(\left(d\!\left(\iota_{X_f}\!\left(d\!\left(g\right)\right)\right) + \iota_{X_f}\!\left(d\!\left(d\!\left(g\right)\right)\right)\right) - \left(\iota_{X_g}\!\left(d\!\left(\iota_{X_f}\!\left(\omega\right)\right)\right) + \iota_{X_g}\!\left(\iota_{X_f}\!\left(d\!\left(\omega\right)\right)\right)\right)\right) - d\!\left(X_f\!\left(g\right)\right) && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\iota_{X_f}\!\left(d\!\left(g\right)\right) &\to X_f\!\left(g\right) && \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(df) = X(f)} \\
d\!\left(d\!\left(g\right)\right) &\to 0 && \text{[d² = 0]\,(axiom)}\;\text{--- apply axiom: d² = 0} \\
\iota_{X_f}\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_{X_f}\!\left(\omega\right) &\to d\!\left(f\right) && \text{[\ensuremath{\iota}\_{X\_f} \ensuremath{\omega} = df]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_{X\_f} \ensuremath{\omega} = df} \\
d\!\left(d\!\left(f\right)\right) &\to 0 && \text{[d² = 0]\,(axiom)}\;\text{--- apply axiom: d² = 0} \\
\iota_{X_g}\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
d\!\left(\omega\right) &\to 0 && \text{[d\ensuremath{\omega} = 0 (symplectic)]\,(axiom)}\;\text{--- apply axiom: d\ensuremath{\omega} = 0 (symplectic)} \\
\iota_{X_f}\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_{X_g}\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\left(\left(d\!\left(X_f\!\left(g\right)\right) + 0\right) - \left(0 + 0\right)\right) - d\!\left(X_f\!\left(g\right)\right) &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{align*}

## 6. Adım adım

Zincirin akışı:

1. **A6 `lie_iota`** — LHS'ın `Sum(L_Xf ι_Xg ω, -ι_Xg L_Xf ω)` olarak açılması.
2. **A3** — ilk terim içindeki $\iota_{X_g}\omega$'nın $dg$'ye düşmesi.
3. **Cartan magic (L_Xf d(g))** — ilk terim $L_{X_f}(dg)$ Cartan ile $(d\iota_{X_f} + \iota_{X_f} d)(dg)$'ye açılır.
4. **Cartan magic (L_Xf ω)** — ikinci terimdeki $L_{X_f}\omega$ Cartan ile açılır.
5. **product-rule** — kompozisyon Act'ları element-level formuna indirgenir.
6-8. **Pairing, $d^2=0$, $\iota_X(0)=0$** — ilk terim $d(X_f(g))$'ye yakınsar.
9-14. **A4, $d^2=0$, A8, $\iota_X(0)=0$** — ikinci terim $0$'a üst üste düşer.
15. **simplify** — kanonik form kapanışı: $d(X_f(g)) + 0 + (-0) + (-d(X_f(g))) \to 0$.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" ↦  {step.after}")
    print()

[1] ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω [axiom]
    ι_[X_f,X_g](ω)
 ↦  (L_X_f(ι_X_g(ω)) + (-ι_X_g(L_X_f(ω))))

[2] ι_{X_g} ω = dg [axiom]
    ι_X_g(ω)
 ↦  d(g)

[3] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_f(d(g))
 ↦  ((d * ι_X_f)(d(g)) + (ι_X_f * d)(d(g)))

[4] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_f(ω)
 ↦  ((d * ι_X_f)(ω) + (ι_X_f * d)(ω))

[5] product-rule 
    ((((d * ι_X_f)(d(g)) + (ι_X_f * d)(d(g))) + (-ι_X_g(((d * ι_X_f)(ω) + (ι_X_f * d)(ω))))) + (-d(X_f(g))))
 ↦  (((d(ι_X_f(d(g))) + ι_X_f(d(d(g)))) + (-(ι_X_g(d(ι_X_f(ω))) + ι_X_g(ι_X_f(d(ω)))))) + (-d(X_f(g))))

[6] ι_X(df) = X(f) [axiom]
    ι_X_f(d(g))
 ↦  X_f(g)

[7] d² = 0 [axiom]
    d(d(g))
 ↦  0

[8] ι_X(f) = 0 on 0-forms [axiom]
    ι_X_f(0)
 ↦  0

[9] ι_{X_f} ω = df [axiom]
    ι_X_f(ω)
 ↦  d(f)

[10] d² = 0 [axiom]
    d(d(f))
 ↦  0

[11] ι_X(f) = 0 on 0-forms [axiom]
    ι_X_g(0)
 ↦  0

[12] dω = 0 (symplectic) [axiom]
    d(ω)
 ↦  0

[13] ι_X(f) = 0 on 0-forms [axiom]
    

## 7. Operator-level kapanış — non-degeneracy

Engine zinciri **element-level** eşitliği verdi:

$$
\iota_{[X_f, X_g]}\omega = d(X_f(g)). \qquad(\star)
$$

(b)'den $\{f, g\} = X_f(g)$, dolayısıyla $d\{f, g\} = d(X_f(g))$. $X_{\{f, g\}}$'in tanımı — Hamiltonian defining relation — $\iota_{X_{\{f,g\}}}\omega = d\{f, g\}$. Bunu ($\star$) ile zincirleyip ortak tarafı yok sayarsak:

$$
\iota_{[X_f, X_g]}\omega = \iota_{X_{\{f, g\}}}\omega.
$$

$\omega$ **non-degenerate** — yani $\omega^{\flat}: TM \to T^*M$ fiber-wise bijektif. Bu, $Y \mapsto \iota_Y \omega$ haritasını enjektif yapıyor: iki vektör alanının $\omega$-contraction'ları eşitse, alanlar da eşit. Dolayısıyla

$$
\boxed{\; [X_f, X_g] = X_{\{f, g\}}. \;}
$$

Bu son "enjektiflik" adımı şu anda engine'de değil — non-degeneracy operator-level bir aksiyom (Faz 12'nin `MusicalCompatibility`'sinin bir köşesi) ve iki vektör alanının kendisini doğrudan eşitlemek `AlternatingForm`-aware dispatch gerektiriyor. Bu notebook pass'i ($\iota$-contraction'a indirip non-degeneracy'i markdown'da çağırma), Faz 12 landing'e kadar bu ailenin standart paterni.

## Sonuç

- **Engine ile kapatılan**: $\iota_{[X_f, X_g]}\omega = d(X_f(g))$ — 15-adımlı zincir, dört problem-aksiyomu + `default_engine` yerleşikleri.
- **Markdown'da kapatılan**: non-degeneracy + (b)'nin sonucu ile operator-level $[X_f, X_g] = X_{\{f, g\}}$.

**Niye iki aşamalı?** İki vektör alanını birbirine eşit kılmak; onları p-form'lara uygulayıp değerleri eşleme dispatch'inde, $\omega$'nun non-degenerate olmasını kullanarak karşılaştırma şartı. Faz 12'nin hedefçilerinden:

- `SymplecticProblem` wrapper — A3, A4, A8, A6 kayıtlarını tek çağrıya indirecek.
- `MusicalCompatibility` rank-p genişlemesi + non-degeneracy property — operator-level $[X_f, X_g] = X_{\{f,g\}}$'yi özel çağrısız kapatabilecek.
- Intrinsik `AlternatingForm` node — $\omega(X, Y)$ node'u için çalışır hale getirip `ω(Y, \cdot)`'nin enjektif oldığu dispatch'i engine-level'a taşıyacak.

Bu üç işi Faz 12.C (ergonomi) + 12.A/B (intrinsik altyapı) üzerinden ayrı talepler olarak yürütüyoruz; bu notebook o iki katmanın **painpoint-örneği**, plan.md §1193-1372'deki listeyi gerekçelendirir.